In [ ]:
%load_ext autoreload
%autoreload 2
from eo_tools.S1.download import download_partial_products, search_products
from eo_tools.util import explore_products
from eo_tools_dev.util import serve_map
from pystac_client.client import Client
import geopandas as gpd
import json

import logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)
from pathlib import Path
import rasterio


In [ ]:
data_dir = "/data/S1/partial_dls/"
out_dir = "/data/res/test_insar_partial/"
# use the creds created on CDSE website
with open("/data/creds_s3.json") as f:
    cred = json.load(f)

## Search products with STAC API

In [ ]:
# Search using STAC api
aoi_file = "../data/Nyamuragira.geojson"
shp = gpd.read_file(aoi_file).geometry[0]

search_args = dict(
    collections=["sentinel-1-slc"],
    intersects=shp,
    datetime=["2026-03-01", "2026-04-30"],
)
products = search_products(**search_args)

In [ ]:
m = explore_products(products=products, aoi=shp)
serve_map(m)

In [ ]:
products[products.relativeOrbitNumber == 174]

In [ ]:
# selecting recent pair
sel = products[products.relativeOrbitNumber == 174].iloc[[0, 4]]
ids = sel.id.values
print(sel.id.values)

In [ ]:
download_partial_products(sel, shp, out_dir=data_dir ,aws_key=cred["username"], aws_secret=cred["password"])

## Inspect partial SAFE products


In [ ]:
partial_products = sorted(Path(data_dir).glob("*.partial.SAFE"))
print(f"Found {len(partial_products)} partial products")
for product_dir in partial_products:
    print(product_dir)

product_dir = partial_products[0]
manifest_file = product_dir / "partial_download.yml"
print(f"\nInspecting: {product_dir.name}")
print(f"Manifest: {manifest_file}")
print(manifest_file.read_text())


In [ ]:
measurement_tiffs = sorted((product_dir / "measurement").glob("*.tiff"))
print(f"Found {len(measurement_tiffs)} cropped measurement TIFFs")

for tiff_file in measurement_tiffs:
    with rasterio.open(tiff_file) as src:
        gcps, gcp_crs = src.gcps
        print("-" * 80)
        print(tiff_file.name)
        print(f"  driver: {src.driver}")
        print(f"  size: {src.width} x {src.height}")
        print(f"  bands: {src.count}")
        print(f"  dtypes: {src.dtypes}")
        print(f"  compression: {src.profile.get('compress')}")
        print(f"  crs: {src.crs}")
        print(f"  transform: {src.transform}")
        print(f"  gcp count: {len(gcps)}")
        print(f"  gcp crs: {gcp_crs}")
        print(f"  dataset tag namespaces: {src.tag_namespaces()}")
        print(f"  dataset tags: {src.tags()}")
        for band_idx in src.indexes:
            print(f"  band {band_idx} tag namespaces: {src.tag_namespaces(band_idx)}")
            print(f"  band {band_idx} tags: {src.tags(band_idx)}")


In [ ]:
# Quick data read from one cropped TIFF
sample_tiff = measurement_tiffs[0]
with rasterio.open(sample_tiff) as src:
    arr = src.read(1, window=rasterio.windows.Window(0, 0, min(src.width, 256), min(src.height, 256)))

print(sample_tiff.name)
print(arr.shape, arr.dtype)
print(f"finite pixels: {arr.size - (~(arr == arr)).sum()} / {arr.size}")